# MPF-Net (Mask-Pooled Fusion Network) on the 3 BRAIN-CNet datasets (D1, D2, D3)

EfficientNet-B0 + U-Net (scSE) + 3 deep-supervision heads + soft-mask fusion classifier.
Seg loss: Tversky(0.7)+BCE. Class loss: CE (label smoothing 0.1). Loss balance: learned uncertainty weights.
AdamW + cosine + EMA, 35 epochs, early stopping. No ASPP / TTA / Grad-CAM.

**Protocol:** stratified 20% hold-out test set (paper's 80/20 split) -> 5-fold CV on the remaining 80%
(the fold's val part is only used for early stopping). Reported: per-fold mean+-std on the test set
(comparable to the paper) and the 5-fold ensemble on the same test set.

**How to use on Kaggle:** Add the dataset(s) via *Add Input*, turn Internet **On** (ImageNet weights + pip),
GPU on, then set `DATASET` in cell 2 and run all. Run one dataset per session.

| Key | Kaggle dataset |
|---|---|
| D1 | atikaakter11/brain-tumor-segmentation-dataset |
| D2 | shubhamgajjar/brain-tumor-classification-2d |
| D3 | wajeehaldeenaljunaid/mixed-brain-tumor-dataset |

In [ ]:
import subprocess, sys
try:
    import segmentation_models_pytorch as smp
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "segmentation-models-pytorch"])
    import segmentation_models_pytorch as smp

import os, re, copy, math, time, json, random, inspect
import numpy as np, cv2, torch, torch.nn as nn, torch.nn.functional as F
import albumentations as A
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from tqdm.auto import tqdm
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", dev, "| smp", smp.__version__, "| torch", torch.__version__)

In [ ]:
# ---------------- CONFIG ----------------
DATASET = "D1"      # "D1" | "D2" | "D3"
SMOKE   = False     # True = tiny quick test run (100 images, 2 epochs, 2 folds)

DATASETS = {
    "D1": dict(folder="brain-tumor-segmentation-dataset", size=512, class_weights=False,
               classes=["normal", "glioma", "meningioma", "pituitary"],
               paper=dict(n=4237, acc=.969, f1=.966, dice=.929)),
    "D2": dict(folder="brain-tumor-classification-2d", size=224, class_weights=True,   # imbalanced (neurocytoma=269)
               classes=["astrocytoma", "glioblastoma", "gliomas", "meningioma", "neurocytoma"],   # this copy has no Normal class
               paper=dict(n=3670, acc=.994, f1=.984, dice=.923)),
    "D3": dict(folder="mixed-brain-tumor-dataset", size=512, class_weights=False,
               classes=["normal", "glioma", "meningioma", "pituitary"],
               paper=dict(n=6380, acc=.982, f1=.982, dice=.875)),
}
CFG = DATASETS[DATASET]
HP = dict(epochs=35, patience=8, batch=16, lr=2e-4, wd=1e-4, ema=0.999,
          n_folds=5, test_frac=0.2, seed=42, workers=2)
if SMOKE: HP.update(epochs=2, n_folds=2, batch=8)

# If class cannot be read from the path (e.g. folders named 1/2/3), fill this: {"folder_name": "class"}
_D13 = {"0": "normal", "1": "glioma", "2": "meningioma", "3": "pituitary"}
OVERRIDES = {"D1": _D13, "D3": _D13,
             "D2": {"Astrocytoma": "astrocytoma", "Glioblastoma": "glioblastoma", "Gliomas": "gliomas",
                    "Meningioma": "meningioma", "Neurocytoma": "neurocytoma"}}
CLASS_OVERRIDE = OVERRIDES[DATASET]

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all(HP["seed"])
CLASSES = CFG["classes"]; NC = len(CLASSES); SIZE = CFG["size"]
OUT = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."

In [ ]:
# ---------------- locate dataset + show its layout (check this once!) ----------------
def find_root(folder):
    for dp, dn, _ in os.walk("/kaggle/input"):
        if os.path.basename(dp) == folder:
            return dp
    return "/kaggle/input"
ROOT = find_root(CFG["folder"])
print("ROOT:", ROOT)
for dp, dn, fs in os.walk(ROOT):
    depth = dp[len(ROOT):].count(os.sep)
    if depth <= 3:
        print("  " * depth + os.path.basename(dp) + f"/  ({len(fs)} files)  e.g. {fs[:2]}")

In [ ]:
# ---------------- build (image, mask, label) index ----------------
IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")
KEYS = [("normal", ("notumor", "no_tumor", "no tumor", "no-tumor", "normal", "healthy", "nontumor", "non-tumor", "notr")),
        ("glioma", ("glioma", "astrocytoma", "glioblastoma", "gltr")),
        ("meningioma", ("meningioma", "metr")),
        ("pituitary", ("pituitary", "pitr")),
        ("neurocytoma", ("neurocytoma",))]

def class_from_path(rel):
    for part in rel.replace("\\", "/").split("/"):
        if part in CLASS_OVERRIDE: return CLASS_OVERRIDE[part]
    s = rel.lower()
    for name, kws in KEYS:
        if any(k in s for k in kws): return name
    return None

def is_mask(rel):
    parts = rel.lower().replace("\\", "/").split("/")
    return any("mask" in p for p in parts) or any(p in ("gt", "label", "labels", "seg", "segmentation") for p in parts)

def norm_stem(f):
    s = os.path.splitext(f)[0].lower()
    s = re.sub(r"[-_ ]m$", "", s)                     # D1 masks are named <image>_m.jpg
    s = re.sub(r"(mask|seg|label|gt)", "", s)
    return re.sub(r"[\W_]+", "", s)

def build_index(root):
    imgs, msks = [], []
    for dp, _, fs in os.walk(root):
        for f in fs:
            if not f.lower().endswith(IMG_EXT): continue
            p = os.path.join(dp, f); rel = os.path.relpath(p, root)
            if any(pt in CFG.get("exclude", []) for pt in rel.lower().replace("\\", "/").split("/")): continue
            (msks if is_mask(rel) else imgs).append((p, rel, f))
    m_by_cls, m_by_stem = {}, {}
    for p, rel, f in msks:
        c = class_from_path(rel); k = norm_stem(f)
        m_by_cls.setdefault((c, k), p); m_by_stem.setdefault(k, p)
    samples, dropped, unknown = [], 0, []
    for p, rel, f in imgs:
        c = class_from_path(rel)
        if c is None: unknown.append(rel); continue
        if c not in CLASSES: dropped += 1; continue
        k = norm_stem(f)
        mp = m_by_cls.get((c, k))                      # same-class match only (no cross-class fallback)
        samples.append(dict(img=p, mask=mp, y=CLASSES.index(c)))
    if unknown:
        raise RuntimeError(f"{len(unknown)} images with unknown class, e.g. {unknown[:5]}. "
                           f"Fill CLASS_OVERRIDE using the folder names printed above.")
    HINT = {"no": "normal", "gl": "glioma", "me": "meningioma", "pi": "pituitary"}   # sanity check vs filename prefix
    chk = bad = 0
    for s_ in samples:
        m_ = re.match(r"[a-z]{1,3}-(no|gl|me|pi)", os.path.basename(s_["img"]).lower())
        if m_: chk += 1; bad += HINT[m_.group(1)] != CLASSES[s_["y"]]
    print(f"filename-vs-folder check: {chk} files checked, {bad} disagree (should be 0)")
    print(f"images={len(samples)} (dropped other-class={dropped}) | mask files found={len(msks)}")
    return samples

samples = build_index(ROOT)
cnt = np.bincount([s["y"] for s in samples], minlength=NC)
print("class counts:", dict(zip(CLASSES, cnt)), "| paper total:", CFG["paper"]["n"])
print("images with a matched mask:", sum(s["mask"] is not None for s in samples))
if SMOKE:
    random.shuffle(samples); samples = samples[:100]

In [ ]:
# ---------------- load everything once into RAM (resized) ----------------
def load_all(samples, size):
    n = len(samples)
    X = np.zeros((n, size, size), np.uint8); M = np.zeros((n, size, size), np.uint8)
    y = np.array([s["y"] for s in samples]); has = np.zeros(n, bool)
    for i, s in enumerate(tqdm(samples, desc="loading")):
        im = cv2.imread(s["img"], cv2.IMREAD_GRAYSCALE)
        X[i] = cv2.resize(im, (size, size), interpolation=cv2.INTER_LINEAR)
        if s["mask"] is not None:
            m = cv2.imread(s["mask"], cv2.IMREAD_GRAYSCALE)
            m = cv2.resize(m, (size, size), interpolation=cv2.INTER_NEAREST)
            M[i] = (m > (127 if m.max() > 1 else 0)); has[i] = True
        elif CLASSES[s["y"]] == "normal":
            has[i] = True                      # healthy slice -> empty mask
    return X, M, y, has

X, M, Y, HAS = load_all(samples, SIZE)
print(X.shape, "| samples with mask supervision:", HAS.sum(), f"({HAS.mean():.0%})")
if HAS.sum() == 0:
    print("WARNING: no masks found -> segmentation loss/Dice disabled (classification only).")

AUG = A.Compose([A.HorizontalFlip(p=0.5),
                 A.Rotate(limit=15, border_mode=cv2.BORDER_CONSTANT, p=0.5),
                 A.RandomBrightnessContrast(0.15, 0.15, p=0.5)])   # applied jointly to image + mask

class DS(Dataset):
    def __init__(s, idx, train): s.idx, s.train = idx, train
    def __len__(s): return len(s.idx)
    def __getitem__(s, i):
        j = s.idx[i]; x, m = X[j], M[j]
        if s.train:
            r = AUG(image=x, mask=m); x, m = r["image"], r["mask"]
        x = torch.from_numpy(np.ascontiguousarray(x)).float().div(255).sub(0.449).div(0.226).unsqueeze(0)
        m = torch.from_numpy(np.ascontiguousarray(m)).float().unsqueeze(0)
        return x, m, int(Y[j]), bool(HAS[j])

def loader(idx, train, bs=None):
    return DataLoader(DS(idx, train), batch_size=bs or HP["batch"], shuffle=train, drop_last=train,
                      num_workers=HP["workers"], pin_memory=True, persistent_workers=HP["workers"] > 0)

In [ ]:
# ---------------- MPFNet ----------------
class MPFNet(nn.Module):
    def __init__(self, num_classes, pretrained=True, enc="tu-efficientnet_b0"):
        super().__init__()
        dch = (256, 128, 64, 32, 16)
        base = smp.Unet(enc, encoder_weights=True if pretrained else None, in_channels=3, classes=1,
                        decoder_channels=dch, decoder_attention_type="scse")
        self.encoder, self.decoder, self.seg_head = base.encoder, base.decoder, base.segmentation_head
        self.aux_ids = (1, 2, 3)                                   # 3 auxiliary 1x1 heads
        self.aux_heads = nn.ModuleList([nn.Conv2d(dch[i], 1, 1) for i in self.aux_ids])
        self._f = {}
        for i in self.aux_ids:
            self.decoder.blocks[i].register_forward_hook(self._mk(i))
        ps = inspect.signature(self.decoder.forward).parameters
        self._star = any(p.kind == p.VAR_POSITIONAL for p in ps.values())   # smp-version compatibility
        c32, c16 = self.encoder.out_channels[-1], self.encoder.out_channels[-2]
        self.head = nn.Sequential(nn.Linear(c32 * 2 + c16, 256), nn.ReLU(True), nn.Dropout(0.3),
                                  nn.Linear(256, num_classes))

    def _mk(self, i):
        def hook(mod, inp, out): self._f[i] = out
        return hook

    def forward(self, x):
        feats = self.encoder(x)
        d = self.decoder(*feats) if self._star else self.decoder(feats)
        seg = self.seg_head(d)
        aux = [h(self._f[i]) for h, i in zip(self.aux_heads, self.aux_ids)]
        m = torch.sigmoid(seg.float())                              # soft mask: no threshold, no detach
        f32, f16 = feats[-1].float(), feats[-2].float()             # stride 32 and stride 16
        def wpool(f):
            mm = F.interpolate(m, size=f.shape[-2:], mode="bilinear", align_corners=False)
            return (f * mm).sum((2, 3)) / (mm.sum((2, 3)) + 1e-6)
        z = torch.cat([f32.mean((2, 3)), wpool(f32), wpool(f16)], 1)
        return dict(seg=seg, aux=aux, cls=self.head(z))

# ---------------- losses ----------------
def tversky(logit, t, alpha=0.7, eps=1.0):
    p = torch.sigmoid(logit.float())
    tp = (p * t).sum(); fn = ((1 - p) * t).sum(); fp = (p * (1 - t)).sum()
    return 1 - (tp + eps) / (tp + alpha * fn + (1 - alpha) * fp + eps)

def seg_loss(out, tgt, valid):
    if valid.sum() == 0: return None
    tot = 0; outs = [out["seg"]] + out["aux"]
    for o in outs:
        o, t = o[valid], tgt[valid]
        if o.shape[-2:] != t.shape[-2:]:
            o = F.interpolate(o.float(), size=t.shape[-2:], mode="bilinear", align_corners=False)
        tot = tot + tversky(o, t) + F.binary_cross_entropy_with_logits(o.float(), t)
    return tot / len(outs)

class Uncertainty(nn.Module):                                      # learned task weights (Kendall et al.)
    def __init__(s): super().__init__(); s.s = nn.Parameter(torch.zeros(2))
    def forward(s, ls, lc):
        t = torch.exp(-s.s[1]) * lc + s.s[1]
        if ls is not None: t = t + torch.exp(-s.s[0]) * ls + s.s[0]
        return t

class EMA:
    def __init__(s, model, decay):
        s.m = MPFNet(NC, pretrained=False).to(dev); s.m.load_state_dict(model.state_dict()); s.m.eval()
        for p in s.m.parameters(): p.requires_grad_(False)
        s.d, s.n = decay, 0
    @torch.no_grad()
    def update(s, model):
        s.n += 1; d = min(s.d, (1 + s.n) / (10 + s.n))
        for e, p in zip(s.m.state_dict().values(), model.state_dict().values()):
            if e.dtype.is_floating_point: e.mul_(d).add_(p.detach(), alpha=1 - d)
            else: e.copy_(p)

In [ ]:
# ---------------- evaluation ----------------
def cls_metrics(y, prob):
    pred = prob.argmax(1)
    p, r, f, _ = precision_recall_fscore_support(y, pred, average="macro", zero_division=0)
    return dict(acc=accuracy_score(y, pred), prec=p, rec=r, f1=f)

def dice_imgs(mprob, gt, has):                                     # per-image Dice, images with non-empty GT
    pred = (mprob > 0.5); gt = gt > 0.5
    inter = (pred & gt).sum((1, 2)); den = pred.sum((1, 2)) + gt.sum((1, 2))
    ok = has & (gt.sum((1, 2)) > 0)
    return (2 * inter[ok] / (den[ok] + 1e-6))

@torch.no_grad()
def predict(model, dl, keep_masks=False):
    model.eval(); P, Ds, MP = [], [], []
    for x, m, y, has in dl:
        x = x.to(dev, non_blocking=True).expand(-1, 3, -1, -1)
        with torch.autocast(dev.type, enabled=dev.type == "cuda"):
            out = model(x)
        P.append(F.softmax(out["cls"].float(), 1).cpu())
        sm = torch.sigmoid(out["seg"].float())[:, 0].cpu()
        Ds.append(dice_imgs(sm, m[:, 0], has))
        if keep_masks: MP.append(sm.half())
    P = torch.cat(P).numpy(); Ds = torch.cat(Ds).numpy()
    return P, Ds, (torch.cat(MP) if keep_masks else None)

def evaluate(model, idx, keep_masks=False):
    P, Ds, MP = predict(model, loader(idx, False, HP["batch"] * 2), keep_masks)
    r = cls_metrics(Y[idx], P); r["dice"] = float(Ds.mean()) if len(Ds) else float("nan")
    return r, P, MP

In [ ]:
# ---------------- one fold ----------------
def train_fold(fold, tr_idx, va_idx):
    seed_all(HP["seed"] + fold)
    model = MPFNet(NC).to(dev); ema = EMA(model, HP["ema"]); unc = Uncertainty().to(dev)
    cw = None
    if CFG["class_weights"]:
        c = np.bincount(Y[tr_idx], minlength=NC); cw = torch.tensor(len(tr_idx) / (NC * np.maximum(c, 1)), dtype=torch.float32, device=dev)
    ce = nn.CrossEntropyLoss(weight=cw, label_smoothing=0.1)
    opt = torch.optim.AdamW([dict(params=model.parameters(), weight_decay=HP["wd"]),
                             dict(params=unc.parameters(), weight_decay=0.0, lr=1e-3)], lr=HP["lr"])
    dl = loader(tr_idx, True); steps = HP["epochs"] * len(dl); warm = len(dl)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm
                                              else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, steps - warm))))
    scaler = torch.amp.GradScaler("cuda", enabled=dev.type == "cuda")
    best, bad, path = -1, 0, f"{OUT}/{DATASET}_fold{fold}.pt"
    for ep in range(HP["epochs"]):
        model.train(); t0 = time.time(); tl = 0
        for x, m, y, has in dl:
            x = x.to(dev, non_blocking=True).expand(-1, 3, -1, -1); m = m.to(dev); y = y.to(dev); has = has.to(dev)
            with torch.autocast(dev.type, enabled=dev.type == "cuda"):
                out = model(x)
            ls = seg_loss(out, m, has); lc = ce(out["cls"].float(), y)
            loss = unc(ls, lc)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward()
            scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step(); ema.update(model); tl += loss.item()
        r, _, _ = evaluate(ema.m, va_idx)
        score = r["f1"] + (0 if math.isnan(r["dice"]) else r["dice"])
        print(f"fold{fold} ep{ep+1:02d} loss {tl/len(dl):.4f} | val acc {r['acc']:.4f} f1 {r['f1']:.4f} "
              f"dice {r['dice']:.4f} | w=({unc.s[0].item():.2f},{unc.s[1].item():.2f}) | {time.time()-t0:.0f}s")
        if score > best: best, bad = score, 0; torch.save(ema.m.state_dict(), path)
        else:
            bad += 1
            if bad >= HP["patience"]: print("early stop"); break
    return path

In [ ]:
# ---------------- run: hold-out test + K folds ----------------
all_idx = np.arange(len(Y))
tr_all, te_idx = next(StratifiedShuffleSplit(1, test_size=HP["test_frac"], random_state=HP["seed"]).split(all_idx, Y))
skf = StratifiedKFold(HP["n_folds"], shuffle=True, random_state=HP["seed"])
print(f"train+val={len(tr_all)}  test={len(te_idx)}")

fold_res, paths = [], []
for k, (a, b) in enumerate(skf.split(tr_all, Y[tr_all])):
    path = train_fold(k, tr_all[a], tr_all[b]); paths.append(path)
    mdl = MPFNet(NC, pretrained=False).to(dev); mdl.load_state_dict(torch.load(path, map_location=dev))
    r, _, _ = evaluate(mdl, te_idx); fold_res.append(r)
    print(f"==> fold {k} TEST: " + "  ".join(f"{n} {v:.4f}" for n, v in r.items()))
    del mdl; torch.cuda.empty_cache()

In [ ]:
# ---------------- per-fold mean + 5-fold ensemble ----------------
print(f"\n===== {DATASET}: per-fold mean +- std on the hold-out test set =====")
summary = {}
for key in ["acc", "prec", "rec", "f1", "dice"]:
    v = np.array([r[key] for r in fold_res]); summary[key] = (float(np.nanmean(v)), float(np.nanstd(v)))
    print(f"{key:5s} {summary[key][0]:.4f} +- {summary[key][1]:.4f}")

P_sum, M_sum = 0, 0
for path in paths:
    mdl = MPFNet(NC, pretrained=False).to(dev); mdl.load_state_dict(torch.load(path, map_location=dev))
    P, _, MP = predict(mdl, loader(te_idx, False, HP["batch"] * 2), keep_masks=True)
    P_sum = P_sum + P; M_sum = M_sum + MP.float(); del mdl; torch.cuda.empty_cache()
P_ens = P_sum / len(paths); M_ens = M_sum / len(paths)
ens = cls_metrics(Y[te_idx], P_ens)
d = dice_imgs(M_ens, torch.from_numpy(M[te_idx]).float(), torch.from_numpy(HAS[te_idx]))
ens["dice"] = float(d.mean()) if len(d) else float("nan")
print(f"\n===== {DATASET}: {len(paths)}-fold ENSEMBLE on the same test set =====")
print("  ".join(f"{k} {v:.4f}" for k, v in ens.items()))
p = CFG["paper"]
if DATASET == "D2": print("\nNOTE: D2 here is a 5-class run without Normal; the paper's D2 numbers are 4-class and NOT comparable.")
else: print(f"\npaper (Table 13, {DATASET}): acc {p['acc']}  f1 {p['f1']}  dice {p['dice']}")

json.dump(dict(dataset=DATASET, folds=fold_res, mean_std=summary, ensemble=ens, hp=HP),
          open(f"{OUT}/{DATASET}_results.json", "w"), indent=2)
from sklearn.metrics import confusion_matrix
print("\nEnsemble confusion matrix (rows=true, cols=pred):", CLASSES); print(confusion_matrix(Y[te_idx], P_ens.argmax(1)))

**Notes**
- Dice = mean per-image Dice at threshold 0.5 over test images that have a non-empty ground-truth mask.
  Precision / recall / F1 are macro-averaged.
- `D1`/`D3` need masks; if a dataset has none, the seg loss is skipped automatically and Dice is NaN.
- If you hit CUDA OOM at 512x512, lower `HP["batch"]` to 8.
- Results are saved to `/kaggle/working/<DATASET>_results.json` and fold weights to `<DATASET>_fold*.pt`.